# 001-seg-map-correction — where the segmentation correlator goes wrong

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import CHANNEL_SETS, SegmentationFeatures, load_unet
from uavloc.features.vgg11 import Vgg11Features, load_vgg11
from uavloc.data.drone_frames import load_frames
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator, agreement_map
from uavloc.plotting.correlation import (CORRECTED_COLOURS, class_colours, mark_points, plot_heatmap, plot_overview,
                                         show_patch)
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "001-seg-map-correction"
SEG_RUN = "001.3"
VGG_RUN = "001.1"
NOISE_M = 15.0
DEVICE = "mps"

CASES_PER_GROUP = 2               # worst reachable draws, one per frame, from each of: trained-on, held-out
OVERVIEW_MARGIN_M = 85.0
SEG_SPLITS = "pocs/village-seg-baseline/results-round3.json"

SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
runs = {r["run"]: r for r in load_runs(EXP)}
config = runs[SEG_RUN]["config"]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}
satellite = GeoImage.load(REPO_ROOT / config["sat_image"], LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
heldout = set(json.loads((REPO_ROOT / SEG_SPLITS).read_text())["splits"]["drone_val"])

In [ ]:
def load_records(run_id: str) -> pd.DataFrame:
    records = pd.read_csv(experiment_dir(EXP) / "artifacts" / f"records-{run_id}.csv")
    return records[records["noise_m"] == NOISE_M].set_index(["frame", "bearing_deg"])

In [ ]:
draws = load_records(SEG_RUN).join(load_records(VGG_RUN)[["error_m"]], rsuffix="_vgg11").reset_index()
draws["group"] = np.where(draws["frame"].isin(heldout), "held-out", "trained-on")

In [ ]:
def select_cases() -> pd.DataFrame:
    """Worst reachable draws per group, at most one per frame: failures the search window cannot excuse."""
    reachable = draws[draws["reachable"]].sort_values("error_m", ascending=False)
    worst_per_frame = reachable.drop_duplicates("frame")
    return pd.concat([g.head(CASES_PER_GROUP) for _, g in worst_per_frame.groupby("group")]).reset_index(drop=True)

In [ ]:
cases = select_cases()
cases[["frame", "group", "bearing_deg", "error_m", "error_m_vgg11", "peak_z"]]

# Model

In [ ]:
seg_features = SegmentationFeatures(load_unet(REPO_ROOT / config["seg_checkpoint_drone"], DEVICE),
                                    load_unet(REPO_ROOT / config["seg_checkpoint_sat"], DEVICE), satellite,
                                    config["seg_channels"], config["drone_seg_gsd_m"], config["match_gsd_m"], DEVICE)
vgg11_features = Vgg11Features(load_vgg11(REPO_ROOT / config["vgg11_checkpoint"], DEVICE), satellite,
                               config["drone_input_px"], config["search_scale"], DEVICE)
correlator_config = CorrelatorConfig(config["search_scale"], config["fov_w_deg"],
                                     sat_feature_crop_pct=config["sat_feature_crop_pct"],
                                     drone_feature_crop_pct=config["drone_feature_crop_pct"])

In [ ]:
captured = {}
seg = PositionCorrelator(seg_features, correlator_config, features_hook=lambda d, s: captured.update(drone=d, satellite=s))
vgg11 = PositionCorrelator(vgg11_features, correlator_config)
class_ids = list(CHANNEL_SETS[config["seg_channels"]])

# Run

In [ ]:
def trace_case(case: pd.Series) -> dict:
    """Replay one logged draw; everything the plots need, in the patch frame of the prior."""
    frame, prior = frames[case["frame"]], LatLon(case["prior_lat"], case["prior_lon"])
    correction = seg.correct(frame, prior)
    footprint_m = seg.footprint_m(frame)
    return {
        "case": case, "frame": frame, "prior": prior, "correction": correction,
        "drone_map": captured["drone"], "satellite_map": captured["satellite"],
        "vgg11_position": vgg11.correct(frame, prior).position, "footprint_m": footprint_m,
        "patch_rgb": satellite.crop_rotated(prior, frame.yaw_deg, *seg.patch_size_m(frame)),
        "true_offset_m": seg.patch_offset_m(prior, frame.position, frame.yaw_deg),
        "at_truth": seg_features.satellite_patch(frame.position, frame.yaw_deg, *footprint_m),
        "at_answer": seg_features.satellite_patch(correction.position, frame.yaw_deg, *footprint_m),
    }

In [ ]:
traces = [trace_case(case) for _, case in cases.iterrows()]
[(t["case"]["frame"], round(haversine_m(t["correction"].position, t["frame"].position), 2), round(t["case"]["error_m"], 2))
 for t in traces]

In [ ]:
def row_label(t: dict) -> str:
    c = t["case"]
    return f"{c['frame'][-7:]} ({c['group']})\nalt {t['frame'].altitude_m:.0f} m  yaw {t['frame'].yaw_deg:.0f}\nerror {c['error_m']:.1f} m"

# Step 1: where each method put the drone

In [ ]:
fig, axes = plt.subplots(1, len(traces), figsize=(5.5 * len(traces), 5.5))
for ax, t in zip(axes, traces):
    plot_overview(ax, satellite, t["frame"], t["prior"], {"round 3": t["correction"].position, "vgg11": t["vgg11_position"]},
                  seg.patch_size_m(t["frame"]), OVERVIEW_MARGIN_M)
    ax.set_title(row_label(t).replace("\n", "  "), fontsize=8)
plt.tight_layout(); plt.show()

# Step 2: drone frame and satellite patch

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(15, 3.6 * len(traces)))
for row, t in zip(axes, traces):
    row[0].imshow(t["frame"].image()); row[0].set_xticks([]); row[0].set_yticks([]); row[0].set_ylabel(row_label(t), fontsize=8)
    show_patch(row[1], t["patch_rgb"], t["correction"].patch_size_m, "satellite patch around the prior")
    mark_points(row[1], t["correction"], t["true_offset_m"], CORRECTED_COLOURS[0])
axes[0, 1].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

# Step 3: drone segmentation

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(12, 3.2 * len(traces)))
for row, t in zip(axes, traces):
    row[0].imshow(t["frame"].image()); row[0].set_ylabel(row_label(t), fontsize=8)
    row[1].imshow(class_colours(t["drone_map"], class_ids)); row[1].set_title("drone classes (what the correlator matches)", fontsize=9)
    for ax in row: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# Step 4: satellite segmentation

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(15, 3.6 * len(traces)))
for row, t in zip(axes, traces):
    size_m = t["correction"].patch_size_m
    show_patch(row[0], t["patch_rgb"], size_m, "satellite patch"); row[0].set_ylabel(row_label(t), fontsize=8)
    show_patch(row[1], class_colours(t["satellite_map"], class_ids), size_m, "satellite classes")
    for ax in row: mark_points(ax, t["correction"], t["true_offset_m"], CORRECTED_COLOURS[0])
plt.tight_layout(); plt.show()

# Step 5: correlation heatmap

In [ ]:
fig, axes = plt.subplots(len(traces), 1, figsize=(8, 3.8 * len(traces)))
for ax, t in zip(axes, traces):
    shown = plot_heatmap(ax, t["patch_rgb"], t["correction"], t["true_offset_m"], CORRECTED_COLOURS[0],
                         f"peak z {t['correction'].peak_z:.2f}")
    ax.set_ylabel(row_label(t), fontsize=8); fig.colorbar(shown, ax=ax, shrink=.8, label="z-score")
plt.tight_layout(); plt.show()

# Step 6: why the wrong place scores higher

In [ ]:
fig, axes = plt.subplots(len(traces), 5, figsize=(24, 2.9 * len(traces)))
for row, t in zip(axes, traces):
    agree_truth, agree_answer = agreement_map(t["drone_map"], t["at_truth"]), agreement_map(t["drone_map"], t["at_answer"])
    panels = [(class_colours(t["drone_map"], class_ids), "drone classes"),
              (class_colours(t["at_truth"], class_ids), "satellite under TRUE footprint"),
              (class_colours(t["at_answer"], class_ids), "satellite under CHOSEN footprint"),
              (agree_truth, f"agreement at truth: {agree_truth.sum():.0f}"),
              (agree_answer, f"agreement at answer: {agree_answer.sum():.0f}")]
    for ax, (image, title) in zip(row, panels):
        ax.imshow(image, cmap="RdYlGn", vmin=0, vmax=1); ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(row_label(t), fontsize=8)
plt.tight_layout(); plt.show()